# FWI (Wildfire Fire Weather Index) — Run Guide

**Collection ID:** `EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI`
**Owner:** Suraj Polade (FMI)

Fire Weather Index (FWI) outputs from DestinE Climate DT simulations — daily FWI
fields, annual statistics, and regional daily climatologies. Historical (1990–2014)
and projection (2015–2049), produced from the Climate DT models.

> ⚠️ **Several config fields are unconfirmed.** Unlike the other three apps, no JSON
> reference metadata was provided for FWI — only a description of the staging logic
> from Suraj Polade (FMI). `bbox`, exact `resolution`/`realization` values, license,
> and the provider list are best-effort placeholders marked `TODO confirm` in
> `catalog_config.yaml`. The folder-naming convention and property order *are*
> confirmed, taken from the legacy staging script that used to live in this repo.

This notebook runs the **whole pipeline against a generated FWI example dataset**
into a throwaway sandbox directory, and stops **before** the upload step. No
credentials needed, nothing is pushed to S3.

For the conda environment and `.env` credential setup, see the top-level
[`README.md`](../../README.md). For how FWI compares to the other three apps, see
[`apps/RUN_GUIDE.md`](../RUN_GUIDE.md).

Prefer a plain shell script over a notebook? See [`run_example.sh`](run_example.sh)
for the same three steps with no narration, just commands and a `SOURCE`/`TARGET`
pair to edit at the top.

## Files in this app

| File | Purpose |
|---|---|
| `catalog_config.yaml` | ALL FWI input: collection metadata, paths, simulation keys |
| `staging/stage.py` | Builds the catalog skeleton: `collection.json` + `data/YYYY/MM/<item>/` folders for both monthly and yearly item shapes |
| `staging/move.py` | Links/copies/moves source NetCDF files into the staged folders — auto-discovers model subfolders |
| `staging/update.py` | Generates STAC item metadata for the staged collection |
| `staging/delete_s3_collection_prefix.py` | Maintenance utility to delete a collection prefix from the S3 bucket |
| `staging/check_data.sh` | Optional pre-staging QC: daily file counts per year, annual statistics completeness |
| `run_pipeline.sh` | Wraps `stage.py` + `update.py` + upload into one command (does **not** run `move.py`) |

FWI has **two genuinely different item shapes that coexist**:
- **monthly** items (`data_type=monthly`): one per model × month, built from daily FWI files
- **yearly** items (`data_type=yearly`): one per model × year, gathering that year's three
  annual statistics files (`basic_and_days`, `quantiles`, `dailyclim_<region>`) into a
  **single** item, placed under the `MM=01` subfolder

In [1]:
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path

import yaml

APP_DIR = Path.cwd()
assert (APP_DIR / "catalog_config.yaml").exists(), "run this notebook from apps/fwi/"

COLLECTION_ID = "EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI"
SANDBOX = Path(tempfile.mkdtemp(prefix="fwi_notebook_"))
ENV = {**os.environ, "CATALOG_HOME_DIR": str(SANDBOX)}
print(f"Sandbox: {SANDBOX}")

def run(*args, check=True):
    print("$", " ".join(map(str, args)))
    result = subprocess.run(args, cwd=APP_DIR, env=ENV, capture_output=True, text=True)
    print(result.stdout, end="")
    print(result.stderr, end="")  # stage.py/move.py/update.py log to stderr
    if check:
        result.check_returncode()

Sandbox: /tmp/fwi_notebook_ijvc7qji


## 1. The config file

In [2]:
config = yaml.safe_load((APP_DIR / "catalog_config.yaml").read_text())
print("id:               ", config["id"])
print("license:          ", config["license"], " (TODO confirm with FMI)")
print("item_folder_level:", config["item_folder_level"])
print("property_order:   ", config["property_order"])
print("simulations:")
for sim in config["simulations"]:
    print(" -", sim)
print("overrides:        ", config["overrides"], " (TODO confirm resolution/realization with FMI)")
print("\napp_data_dir:     ", config["app_data_dir"])
print("catalog_home_dir: ", config["catalog_home_dir"], " (overridden to the sandbox for this notebook)")

id:                EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI
license:           CC-BY-4.0  (TODO confirm with FMI)
item_folder_level: MM
property_order:    ['model', 'experiment', 'generation', 'realization', 'resolution', 'activity', 'data_type']
simulations:
 - ifs-nemo_generation-2_historical
 - ifs-nemo_generation-2_projection
 - ifs-fesom_generation-2_historical
 - ifs-fesom_generation-2_projection
 - icon_generation-2_historical
 - icon_generation-2_projection
overrides:         {'realization': 1, 'resolution': '5km'}  (TODO confirm resolution/realization with FMI)

app_data_dir:      /work/lealroja/applications-stac-catalog-phase3-fwi-output-example
catalog_home_dir:  /work/lealroja/applications-stac-catalog-phase3-execution  (overridden to the sandbox for this notebook)


## 2. Step 1 — `stage.py`: build the catalog skeleton

Pure config — no NC files are read. Builds **both** monthly and yearly skeleton
folders for every (model, year) combination: 3 models × 60 years × (12 monthly + 1
yearly) = 2340 folders.

In [3]:
run(sys.executable, "staging/stage.py")  # dry run

$ /gpfs1/schlecker/global/software/skylake/software/Miniforge3/25.3.1-0/bin/python3.12 staging/stage.py


INFO | Collection: EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI (6 simulations)
INFO | [DRY RUN] Would write collection.json + collection_config.json
INFO | [DRY RUN] Skeleton for IFS-NEMO, IFS-FESOM, ICON: would create up to 2340 folders (2160 monthly, 180 yearly)
INFO | Dry run complete. Re-run with --execute to apply.


In [4]:
run(sys.executable, "staging/stage.py", "--execute")

$ /gpfs1/schlecker/global/software/skylake/software/Miniforge3/25.3.1-0/bin/python3.12 staging/stage.py --execute


INFO | Collection: EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI (6 simulations)
INFO | Wrote /tmp/fwi_notebook_ijvc7qji/EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI/metadata/collection.json and /tmp/fwi_notebook_ijvc7qji/EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI/metadata/collection_config.json
INFO | Skeleton for IFS-NEMO, IFS-FESOM, ICON: 2340 folders created (2160 monthly, 180 yearly)


## 3. Step 2 — `check_data.sh` (optional): completeness check

Per model: counts daily FWI files per year (expects 365/366), checks all three
annual statistics files are present, flags unrecognized filenames. Run against the
generated FWI example dataset (2 models, 2 source years, partial — intentionally
incomplete, just enough to demonstrate the pipeline).

In [ ]:
FWI_SOURCE = "/work/lealroja/applications-stac-catalog-phase3-fwi-output-example"  # <-- HARDCODED PATH: change to your own top-level source dir (one subfolder per model) to re-run with different data

result = subprocess.run(
    ["bash", "staging/check_data.sh", FWI_SOURCE],
    cwd=APP_DIR, capture_output=True, text=True,
)
print(result.stdout)

## 4. Step 3 — `move.py`: bring the NC files in

Unlike the other three apps, `--source-dir` is the **top-level** directory
containing one subfolder per model — the model is read from that subfolder name
automatically, no `--model` flag, and all models found are processed in one call.

```
<source-dir>/
├── IFS-NEMO/
│   ├── run01/member01/
│   │   ├── 1995_06_15_T00_00_fwi.nc           ← daily FWI field
│   │   └── 2030_03_10_T00_00_fwi.nc
│   └── Annual_Stat/
│       ├── 1995_fwi_basic_and_days.nc          ← annual statistics (3 files/year)
│       ├── 1995_fwi_quantiles.nc
│       └── 1995_fwi_dailyclim_Europe.nc
└── ICON/
    ├── run01/member01/2000_01_05_T00_00_fwi.nc
    └── Annual_Stat/2000_fwi_basic_and_days.nc
```

`experiment`/`activity` are derived from each file's own year (`historical`/
`baseline` for 1990–2014, `ssp3-7.0`/`projections` for 2015–2049) — not a flag.

In [6]:
move_args = [
    sys.executable, "staging/move.py",
    "--source-dir", FWI_SOURCE,
    "--target-collection-dir", str(SANDBOX / COLLECTION_ID),
    "--generation", "2", "--realization", "1", "--resolution", "5km",
    "--link",
]
run(*move_args)  # dry run

$ /gpfs1/schlecker/global/software/skylake/software/Miniforge3/25.3.1-0/bin/python3.12 staging/move.py --source-dir /work/lealroja/applications-stac-catalog-phase3-fwi-output-example --target-collection-dir /tmp/fwi_notebook_ijvc7qji/EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI --generation 2 --realization 1 --resolution 5km --link


INFO | [symlink] /gpfs1/work/lealroja/applications-stac-catalog-phase3-fwi-output-example/ICON/run01/member01/2000_01_05_T00_00_fwi.nc -> /tmp/fwi_notebook_ijvc7qji/EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI/data/2000/01/EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI_20000101T000000_20000131T230000__ICON__historical__2__1__5km__baseline__monthly/2000_01_05_T00_00_fwi.nc
INFO | [symlink] /gpfs1/work/lealroja/applications-stac-catalog-phase3-fwi-output-example/ICON/Annual_Stat/2000_fwi_basic_and_days.nc -> /tmp/fwi_notebook_ijvc7qji/EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI/data/2000/01/EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI_20000101T000000_20001231T230000__ICON__historical__2__1__5km__baseline__yearly/2000_fwi_basic_and_days.nc
INFO | [symlink] /gpfs1/work/lealroja/applications-stac-catalog-phase3-fwi-output-example/IFS-NEMO/run01/member01/1995_06_15_T00_00_fwi.nc -> /tmp/fwi_notebook_ijvc7qji/EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI/data/1995/06/EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI_19950601T000

In [7]:
run(*move_args, "--execute")

$ /gpfs1/schlecker/global/software/skylake/software/Miniforge3/25.3.1-0/bin/python3.12 staging/move.py --source-dir /work/lealroja/applications-stac-catalog-phase3-fwi-output-example --target-collection-dir /tmp/fwi_notebook_ijvc7qji/EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI --generation 2 --realization 1 --resolution 5km --link --execute


INFO | [symlink] /gpfs1/work/lealroja/applications-stac-catalog-phase3-fwi-output-example/ICON/run01/member01/2000_01_05_T00_00_fwi.nc -> /tmp/fwi_notebook_ijvc7qji/EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI/data/2000/01/EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI_20000101T000000_20000131T230000__ICON__historical__2__1__5km__baseline__monthly/2000_01_05_T00_00_fwi.nc
INFO | [symlink] /gpfs1/work/lealroja/applications-stac-catalog-phase3-fwi-output-example/ICON/Annual_Stat/2000_fwi_basic_and_days.nc -> /tmp/fwi_notebook_ijvc7qji/EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI/data/2000/01/EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI_20000101T000000_20001231T230000__ICON__historical__2__1__5km__baseline__yearly/2000_fwi_basic_and_days.nc
INFO | [symlink] /gpfs1/work/lealroja/applications-stac-catalog-phase3-fwi-output-example/IFS-NEMO/run01/member01/1995_06_15_T00_00_fwi.nc -> /tmp/fwi_notebook_ijvc7qji/EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI/data/1995/06/EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI_19950601T000

## 5. Step 4 — `update.py`: generate STAC item metadata

In [8]:
run(sys.executable, "staging/update.py")

$ /gpfs1/schlecker/global/software/skylake/software/Miniforge3/25.3.1-0/bin/python3.12 staging/update.py


PySTAC version: 1.0.0
Collection passed all field checks.
item_id:EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI_20491201T000000_20491231T230000__ICON__ssp3-7.0__2__1__5km__projections__monthly, collection_id:EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI, year:2049, month:12, day:01
item_id:EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI_20491201T000000_20491231T230000__IFS-FESOM__ssp3-7.0__2__1__5km__projections__monthly, collection_id:EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI, year:2049, month:12, day:01
item_id:EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI_20491201T000000_20491231T230000__IFS-NEMO__ssp3-7.0__2__1__5km__projections__monthly, collection_id:EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI, year:2049, month:12, day:01
item_id:EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI_20491101T000000_20491130T230000__ICON__ssp3-7.0__2__1__5km__projections__monthly, collection_id:EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI, year:2049, month:11, day:01
item_id:EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI_20491101T000000_20491130T230000__

## 6. Inspect the results

Note the merged yearly item below — `1995_fwi_basic_and_days.nc`,
`1995_fwi_quantiles.nc`, and `1995_fwi_dailyclim_Europe.nc` all become assets of the
*same* item.

In [9]:
items_dir = SANDBOX / COLLECTION_ID / "metadata" / "items"
item_paths = sorted(items_dir.glob("*.json"))
print(f"{len(item_paths)} item JSON(s) generated:")
for p in item_paths:
    print(" ", p.name)

7 item JSON(s) generated:
  EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI_19950101T000000_19951231T230000__IFS-NEMO__historical__2__1__5km__baseline__yearly.json
  EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI_19950601T000000_19950630T230000__IFS-NEMO__historical__2__1__5km__baseline__monthly.json
  EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI_19950701T000000_19950731T230000__IFS-NEMO__historical__2__1__5km__baseline__monthly.json
  EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI_20000101T000000_20000131T230000__ICON__historical__2__1__5km__baseline__monthly.json
  EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI_20000101T000000_20001231T230000__ICON__historical__2__1__5km__baseline__yearly.json
  EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI_20300101T000000_20301231T230000__IFS-NEMO__ssp3-7.0__2__1__5km__projections__yearly.json
  EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI_20300301T000000_20300331T230000__IFS-NEMO__ssp3-7.0__2__1__5km__projections__monthly.json


In [10]:
yearly = [p for p in item_paths if "__yearly" in p.name][0]
sample = json.loads(yearly.read_text())
print("id:        ", sample["id"])
print("properties:", sample["properties"])
print("assets:    ", list(sample["assets"]))

id:         EO.FMI.DAT.DESTINE_CLIMATE_WILDFIRE_FWI_19950101T000000_19951231T230000__IFS-NEMO__historical__2__1__5km__baseline__yearly
properties: {'model': 'IFS-NEMO', 'experiment': 'historical', 'generation': '2', 'realization': '1', 'resolution': '5km', 'activity': 'baseline', 'data_type': 'yearly', 'start_datetime': '1995-01-01T00:00:00Z', 'end_datetime': '1995-12-31T23:00:00Z', 'datetime': '1995-01-01T00:00:00Z'}
assets:     ['1995_fwi_basic_and_days.nc', '1995_fwi_dailyclim_Europe.nc', '1995_fwi_quantiles.nc']


## 7. Upload (not run in this notebook)

Skipped here on purpose. To actually publish:

1. Copy `apps/fwi/.env.example` → `apps/fwi/.env` and fill in
   `AWS_ACCESS_KEY_ID` / `AWS_SECRET_ACCESS_KEY` (see the top-level README's
   "Configure credentials" section).
2. Remove the `CATALOG_HOME_DIR` sandbox override, and **confirm the TODO-marked
   fields in `catalog_config.yaml` with FMI first** — `bbox`, `resolution`,
   `realization`, license, and providers are placeholders.
3. Run:
   ```bash
   bash run_pipeline.sh --execute --upload
   bash run_pipeline.sh --execute --upload --year 1995   # resume/retry a single year
   ```

In [11]:
import shutil
shutil.rmtree(SANDBOX)
print(f"Removed sandbox: {SANDBOX}")

Removed sandbox: /tmp/fwi_notebook_ijvc7qji
